# 2차 팀 프로젝트 개발 (2) — `app_v1.py`(초기본) → `app.py`(최종본) 무엇이 바뀌었나

같은 대시보드의 두 버전을 `difflib` 으로 비교해, 발표 전까지 고친 내용을 **분류별로 정리**한 노트입니다.
(계산 로직은 [`streamlit_dashboard_logic.ipynb`](./streamlit_dashboard_logic.ipynb))

In [1]:
import difflib
from pathlib import Path

v1 = Path('team_project/app_v1.py').read_text(encoding='utf-8').splitlines()
v2 = Path('team_project/app.py').read_text(encoding='utf-8').splitlines()
sm = difflib.SequenceMatcher(None, v1, v2)
ops = [op for op in sm.get_opcodes() if op[0] != 'equal']
added = sum(j2 - j1 for tag, i1, i2, j1, j2 in ops if tag in ('insert', 'replace'))
removed = sum(i2 - i1 for tag, i1, i2, j1, j2 in ops if tag in ('delete', 'replace'))
print(f'초기본 {len(v1)}줄 → 최종본 {len(v2)}줄 | 바뀐 덩어리 {len(ops)}개 | 추가 {added}줄 · 삭제 {removed}줄 | 유사도 {sm.ratio():.3f}')

초기본 919줄 → 최종본 967줄 | 바뀐 덩어리 23개 | 추가 91줄 · 삭제 43줄 | 유사도 0.929


In [2]:
def show(keyword, context=0, limit=6):
    """keyword 가 들어간 변경 줄만 보기"""
    n = 0
    for line in difflib.unified_diff(v1, v2, 'app_v1.py', 'app.py', lineterm='', n=context):
        if line.startswith(('+', '-')) and not line.startswith(('+++', '---')) and keyword in line:
            print(line[:150]); n += 1
            if n >= limit:
                break

print('[페이지 이름]'); show('PAGES')
print('\n[입력 폼 · 단위]'); show('number_input', limit=6); show('avg_value'); show('caption')

[페이지 이름]
-PAGES = ["요약", "고객 이탈 예측", "데이터 분석", "모델 성능"]
+PAGES = ["핵심 지표", "고객 이탈 예측", "이탈 원인 분석", "모델 성능"]

[입력 폼 · 단위]
-            points = f2.number_input("보유 포인트", value=None, placeholder="예: 720", step=0.01)
+            points = f2.number_input("보유 포인트 (만원)", value=None, placeholder="예: 720", step=0.01)
-            total_value = f1.number_input("누적 이용 금액", min_value=0.0, value=None,
+            total_value = f1.number_input("누적 이용 금액 (만원)", min_value=0.0, value=None,
-                avg_value = float(total_value) / int(total_count)
+                avg_value = float(total_value) * MANWON / int(total_count)
-            st.caption("평균 거래 금액은 누적 이용 금액 ÷ 누적 이용 횟수로 계산합니다.")
+            st.caption("금액은 만원 단위로 입력합니다. 평균 거래 금액 = 누적 이용 금액 × 10,000 ÷ 누적 이용 횟수")


In [3]:
print('[CSS 애니메이션]'); show('@keyframes', limit=8)
print('\n[레이아웃]'); show('row-grid', limit=4); show('st.columns', limit=4)
print('\n[히어로 문구]'); show('<h1>', limit=4)

[CSS 애니메이션]
+@keyframes si-grow-w{from{width:0;}}
+@keyframes si-grow-h{from{height:0;}}
+@keyframes si-draw{to{stroke-dashoffset:0;}}
+@keyframes si-fade{from{opacity:0;}}
+@keyframes si-pop{from{opacity:0;transform:scale(.85);}}

[레이아웃]
+.hero,.card,.kpi,.crumb,.footer,.info-band,.side-card,.side-meta,.side-status,.brand,.form-h,.row-grid{
+.row-grid{display:grid;grid-template-columns:minmax(0,1.15fr) minmax(0,.85fr);gap:16px;align-items:stretch;margin-bottom:2px;}
+.row-grid > .card,.row-grid > .kpis{margin:0;}
+.row-grid > .kpis > .kpi{height:100%;}
-    c1, c2 = st.columns([1.15, 0.85], gap="medium")
+    #  - st.columns는 칸마다 높이가 따로 정해져 두 카드 높이가 어긋나고, 칸 사이 간격도 시안(16px)보다 넓음

[히어로 문구]
-    <h1>고객이 떠나기 전에,<br>지금 행동하세요.</h1>
+    <h1>데이터가 이끄는<br>이커머스의 새로운 성장</h1>


## 변경 정리

| 분류 | 초기본 `app_v1.py` | 최종본 `app.py` |
|---|---|---|
| 페이지 이름 | 요약 · 고객 이탈 예측 · **데이터 분석** · 모델 성능 | **핵심 지표** · 고객 이탈 예측 · **이탈 원인 분석** · 모델 성능 |
| 첫 화면 KPI | 고객 수 · 이탈 비율 · 입력 특성 4개 · 결측 0개 (4칸) | 고객 수 · 이탈 비율 (2칸) |
| 레이아웃 | `st.columns` (칸마다 높이가 달라 카드가 어긋남) | **CSS grid** (`row-grid`) — 같은 줄 카드 높이 자동 맞춤 |
| 본문 너비 · 글꼴 | 기본 최대 1400px · Source Sans | 화면 전체 너비 · Pretendard (HTML 시안과 같게) |
| 그래프 | 정지 | **CSS 애니메이션** — 막대가 0에서 자라고(`si-grow-w/h`) 선이 그려짐(`stroke-dashoffset`) |
| 입력 카드 | 제목 카드와 폼이 따로 | `st.container(key='predict_card')` 로 **흰 카드 하나** |
| 금액 입력 | 누적 이용 금액(원), 예: 250000 | 누적 이용 금액 **(만원)**, 예: 25 → × 10,000 |
| 포인트 입력 | '보유 포인트' | '보유 포인트 **(만원)**' ← 📝 포인트는 금액이 아님 |
| 히어로 | '고객이 떠나기 전에, 지금 행동하세요.' + 기간 선택 칩 | '데이터가 이끄는 이커머스의 새로운 성장' |

In [4]:
# 금액 단위 변경이 같은 결과를 내는지 : 초기본(원) vs 최종본(만원)
def avg_v1(total_won, count):
    return float(total_won) / int(count)

def avg_v2(total_manwon, count):
    return float(total_manwon) * 10_000 / int(count)

for won, cnt in [(250_000, 10), (1_200_000, 30), (80_000, 2)]:
    print(f'{won:>9,}원 / {cnt:2d}회 → 초기본 {avg_v1(won, cnt):>8,.0f} | 최종본 {won / 10_000:6.1f}만원 → {avg_v2(won / 10_000, cnt):>8,.0f}')

  250,000원 / 10회 → 초기본   25,000 | 최종본   25.0만원 →   25,000
1,200,000원 / 30회 → 초기본   40,000 | 최종본  120.0만원 →   40,000
   80,000원 /  2회 → 초기본   40,000 | 최종본    8.0만원 →   40,000


- 금액 단위 변경은 **계산 결과가 같다**(만원 입력이 자릿수 실수를 줄여 줌). 대신 포인트 입력에도 '(만원)' 이 붙어 버린 것은 **새로 생긴 표기 오류** — 계산은 바뀌지 않았다
- 화면 개선(그리드 · 애니메이션 · 카드)은 계산 로직(`ChurnMLP` · `evaluate_mlp` · `create_analysis_tables`)을 건드리지 않았다 → 아래에서 확인

In [5]:
import re

def func_body(lines, name):
    text = '\n'.join(lines)
    m = re.search(rf'^(def {name}\(|class {name}\b).*?(?=^\S)', text, flags=re.S | re.M)
    return m.group(0) if m else ''

for name in ['ChurnMLP', 'evaluate_mlp', 'create_analysis_tables', 'prepare_model_input', 'quartile_series']:
    b1, b2 = func_body(v1, name), func_body(v2, name)
    print(f'{name:24s} {len(b2):5,}자 · 두 버전 동일 ? {b1 == b2 and len(b1) > 0}')

ChurnMLP                 3,255자 · 두 버전 동일 ? True
evaluate_mlp             1,637자 · 두 버전 동일 ? True
create_analysis_tables   3,338자 · 두 버전 동일 ? True
prepare_model_input        534자 · 두 버전 동일 ? True
quartile_series            249자 · 두 버전 동일 ? True
